In [65]:
import numpy as np
import pandas as pd
import statistics

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_validate, train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)


In [66]:
data = pd.read_csv("../data/data2.csv", index_col=0)
print(data)

           country time_of_day        lat       long      road_type  \
Unnamed: 0                                                            
8459            SE         day  55.604209  13.028574           city   
71801           IT         day  41.987014  12.496538        highway   
90649           DE         day  49.182153   9.414737        highway   
48806           SE         day  55.576732  13.013051           city   
35874           IT         day  45.478483   9.200364           city   
...            ...         ...        ...        ...            ...   
11242           SE         day  55.598127  12.975875           city   
90673           PL         day  50.253217  19.823732  smaller-rural   
24373           DE         day  50.931399   6.953686           city   
41597           SE         day  55.608149  13.003458           city   
40353           IT       night  41.918667  12.383545           city   

           road_condition            weather  solar_angle_elevation  month  

In [67]:
data = data.dropna().reset_index(drop=True)
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 5,985 rows and 38 columns


In [68]:
iou = data["iou"]
lrp = data["lrp"]

data = data.drop(columns=["iou", "lrp", "conf"])
data_indices = data.index.to_numpy()

In [69]:
all_columns = data.columns    
all_columns = all_columns.drop(["weather_code", "cloud_cover_low", "cloud_cover_mid"])

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 5985
Columns: Index(['brightness', 'camera_distance_from_ground', 'camera_offset',
       'camera_pitch_angle', 'cloud_cover', 'complexity', 'contrast',
       'country', 'distortion_magnitude', 'field_view_horizontal',
       'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat',
       'lateral_acceleration', 'lateral_velocity', 'long', 'month',
       'noisiness', 'quality', 'rain', 'relative_humidity_2m',
       'road_condition', 'road_type', 'sharpness', 'snowfall',
       'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather',
       'wind_speed_10m'],
      dtype='object')
Number of columns: 32


In [70]:
numeric_columns = ['solar_angle_elevation', 'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m', 'forward_acceleration', 'lateral_acceleration', 'forward_velocity', 'lateral_velocity', 'field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset', 'laplacian', 'brightness', 'contrast', 'sharpness', 'noisiness', 'quality', 'complexity']

categorical_columns = ["country", "time_of_day", "road_type", "road_condition", "weather"]
other = ["hour", "month", "lat", "long"]
print(sorted(numeric_columns + categorical_columns + other))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns) + len(other)), "Columns not match"

['brightness', 'camera_distance_from_ground', 'camera_offset', 'camera_pitch_angle', 'cloud_cover', 'complexity', 'contrast', 'country', 'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration', 'lateral_velocity', 'long', 'month', 'noisiness', 'quality', 'rain', 'relative_humidity_2m', 'road_condition', 'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather', 'wind_speed_10m']


# Assessor Tests

### Baseline

Random Prediction (Normal dist)

In [71]:
# Collect metrics for IoU and LRP
model_results = []


In [72]:
random_iou = np.random.normal(loc=0.5, scale=0.5, size=len(data))
random_iou = np.clip(random_iou, 0, 1)
random_baseline_r2 = r2_score(iou, random_iou)
random_baseline_mae = np.mean(np.abs(iou - random_iou))
random_baseline_mse = np.mean((iou - random_iou)**2)
print(f"R2 score of random baseline: {random_baseline_r2:.4f}")
print(f"MAE of random baseline: {random_baseline_mae:.4f}")
print(f"MSE of random baseline: {random_baseline_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Random Baseline',
    'test_r2': random_baseline_r2,
    'test_mae': random_baseline_mae,
    'test_mse': random_baseline_mse,
})


R2 score of random baseline: -6.6410
MAE of random baseline: 0.3471
MSE of random baseline: 0.1706


### IoU

Split data into train 60% and validation 40%

In [73]:
#X_train, X_test, y_train, y_test = train_test_split(data, test_size=0.4, train_size=0.6, stratify=data["iou"])
train_idx, test_idx = train_test_split(data_indices, test_size=0.4, train_size=0.6)

X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]


print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print(X_train.columns)

X: 3591 2394
y: 3591 2394
Index(['country', 'time_of_day', 'lat', 'long', 'road_type', 'road_condition',
       'weather', 'solar_angle_elevation', 'month', 'hour',
       'forward_acceleration', 'lateral_acceleration', 'forward_velocity',
       'lateral_velocity', 'field_view_horizontal',
       'camera_distance_from_ground', 'camera_pitch_angle',
       'distortion_magnitude', 'camera_offset', 'laplacian', 'quality',
       'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity',
       'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall',
       'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'wind_speed_10m',
       'weather_code'],
      dtype='object')


Train models with cv and then test.

#### Linear Regression

In [74]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
cv_results = cross_validate(
    linear_reg,
    X_train,
    y_train,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.0437
Mean CV test r2 score -0.0249
Mean CV train MAE score 0.1024
Mean CV test MAE score 0.1049
Mean CV train MSE score 0.0207
Mean CV test MSE score 0.0221


In [75]:
linear_reg.fit(X_train, y_train)
y_pred = linear_reg.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Linear Regression',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score -0.0009
MAE test score 0.1076
MSE test score 0.0233


#### Decision Trees

In [76]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor())
])
cv_results = cross_validate(
    decision_tree,
    X_train,
    y_train,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 1.0000
Mean CV test r2 score -1.0418
Mean CV train MAE score 0.0000
Mean CV test MAE score 0.1463
Mean CV train MSE score 0.0000
Mean CV test MSE score 0.0435


In [77]:
decision_tree.fit(X_train, y_train)
y_pred = decision_tree.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Decision Tree',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score -0.8989
MAE test score 0.1471
MSE test score 0.0442


#### Random Forest

In [78]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor())
])
cv_results = cross_validate(
    rand_forest,
    X_train,
    y_train,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.8525
Mean CV test r2 score -0.0337
Mean CV train MAE score 0.0393
Mean CV test MAE score 0.1053
Mean CV train MSE score 0.0032
Mean CV test MSE score 0.0223


In [79]:
rand_forest.fit(X_train, y_train)
y_pred = rand_forest.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Random Forest',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.0004
MAE test score 0.1080
MSE test score 0.0233


#### MLP

In [80]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor())
])
cv_results = cross_validate(
    mlp,
    X_train,
    y_train,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.0077
Mean CV test r2 score -0.5255
Mean CV train MAE score 0.1077
Mean CV test MAE score 0.1280
Mean CV train MSE score 0.0215
Mean CV test MSE score 0.0329


In [81]:
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'MLP',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score -0.4568
MAE test score 0.1281
MSE test score 0.0339


### LRP


Random Prediction (Normal dist)


In [82]:
random_lrp = np.random.normal(loc=0.5, scale=0.5, size=len(data))
random_lrp = np.clip(random_lrp, 0, 1)
random_lrp_r2 = r2_score(lrp, random_lrp)
random_lrp_mae = np.mean(np.abs(lrp - random_lrp))
random_lrp_mse = np.mean((lrp - random_lrp)**2)
print(f"R2 score of random baseline: {random_lrp_r2:.4f}")
print(f"MAE of random baseline: {random_lrp_mae:.4f}")
print(f"MSE of random baseline: {random_lrp_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Random Baseline',
    'test_r2': random_lrp_r2,
    'test_mae': random_lrp_mae,
    'test_mse': random_lrp_mse,
})


R2 score of random baseline: -10.2705
MAE of random baseline: 0.3615
MSE of random baseline: 0.1897


Split data into train 60% and validation 40%


In [83]:
y_train_lrp = lrp.loc[train_idx]
y_test_lrp = lrp.loc[test_idx]
print('X:', len(X_train), len(X_test))
print('y:', len(y_train_lrp), len(y_test_lrp))


X: 3591 2394
y: 3591 2394


Train models with cv and then test.


#### Linear Regression


In [84]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_lrp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
cv_results = cross_validate(
    linear_reg_lrp,
    X_train,
    y_train_lrp,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.0849
Mean CV test r2 score 0.0227
Mean CV train MAE score 0.0915
Mean CV test MAE score 0.0936
Mean CV train MSE score 0.0152
Mean CV test MSE score 0.0161


In [85]:
linear_reg_lrp.fit(X_train, y_train_lrp)
y_pred = linear_reg_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Linear Regression',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score -0.0196
MAE test score 0.0966
MSE test score 0.0174


#### Decision Trees


In [86]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_lrp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor())
])
cv_results = cross_validate(
    decision_tree_lrp,
    X_train,
    y_train_lrp,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 1.0000
Mean CV test r2 score -1.0492
Mean CV train MAE score -0.0000
Mean CV test MAE score 0.1324
Mean CV train MSE score -0.0000
Mean CV test MSE score 0.0335


In [87]:
decision_tree_lrp.fit(X_train, y_train_lrp)
y_pred = decision_tree_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Decision Tree',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score -0.8988
MAE test score 0.1319
MSE test score 0.0325


#### Random Forest


In [88]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_lrp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor())
])
cv_results = cross_validate(
    rand_forest_lrp,
    X_train,
    y_train_lrp,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.8615
Mean CV test r2 score -0.0058
Mean CV train MAE score 0.0349
Mean CV test MAE score 0.0945
Mean CV train MSE score 0.0023
Mean CV test MSE score 0.0166


In [89]:
rand_forest_lrp.fit(X_train, y_train_lrp)
y_pred = rand_forest_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Random Forest',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score -0.0061
MAE test score 0.0957
MSE test score 0.0172


#### MLP


In [90]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_lrp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor())
])
cv_results = cross_validate(
    mlp_lrp,
    X_train,
    y_train_lrp,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score -0.0238
Mean CV test r2 score -0.6437
Mean CV train MAE score 0.0992
Mean CV test MAE score 0.1183
Mean CV train MSE score 0.0171
Mean CV test MSE score 0.0268


In [91]:
mlp_lrp.fit(X_train, y_train_lrp)
y_pred = mlp_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'MLP',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score -0.4239
MAE test score 0.1127
MSE test score 0.0243


### Final Model Comparison


In [92]:
results_df = pd.DataFrame(model_results, index=None)
results_df["test_r2"] = results_df["test_r2"].round(4)
results_df["test_mae"] = results_df["test_mae"].round(2)
results_df["test_mse"] = results_df["test_mse"].round(2)
display(results_df)


,target,model,test_r2,test_mae,test_mse
0,IoU,Random Baseline,-6.6410,0.35,0.17
1,IoU,Linear Regression,-0.0009,0.11,0.02
2,IoU,Decision Tree,-0.8989,0.15,0.04
3,IoU,Random Forest,0.0004,0.11,0.02
4,IoU,MLP,-0.4568,0.13,0.03
5,LRP,Random Baseline,-10.2705,0.36,0.19
6,LRP,Linear Regression,-0.0196,0.10,0.02
7,LRP,Decision Tree,-0.8988,0.13,0.03
8,LRP,Random Forest,-0.0061,0.10,0.02
9,LRP,MLP,-0.4239,0.11,0.02
